# FMAifs Rollout (Date-Range Controlled)

This notebook runs inference using the FMAifs model over a specified date range.

## What this does

- Loads preprocessed atmospheric states from `.pkl` files
- Runs autoregressive rollouts using a trained model
- Generates future states at fixed lead times
- Saves the resulting trajectory for each initialization time

## Key behavior

- Date-driven processing loop
- Missing files are skipped safely
- Outputs are saved per initialization time

In [1]:
print("FMAifs ROLLOUT")
# import os
# os.environ["ANEMOI_INFERENCE_NUM_CHUNKS"] = "32"
import s3fs
import torch
import pickle
from pathlib import Path
from datetime import datetime, timedelta
import numpy as np
import gc
# from anemoi.inference.runners.simple import SimpleRunner
# from anemoi.inference.outputs.printer import print_state

FMAifs ROLLOUT


In [2]:
import os

os.environ["ANEMOI_INFERENCE_NUM_CHUNKS"] = "32"
os.environ["ANEMOI_INFERENCE_NUM_CHUNKS_MAPPER"] = "32"
os.environ["ANEMOI_INFERENCE_NUM_CHUNKS_PROCESSOR"] = "32"

# optional, helps fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from anemoi.inference.runners.simple import SimpleRunner

In [3]:
import os
print(os.environ.get("ANEMOI_INFERENCE_NUM_CHUNKS"))
print(os.environ.get("ANEMOI_INFERENCE_NUM_CHUNKS_MAPPER"))
print(os.environ.get("ANEMOI_INFERENCE_NUM_CHUNKS_PROCESSOR"))

32
32
32


In [4]:
# -----------------------------
# CONFIG
# -----------------------------

# Assumptions:
# - Input files follow naming convention: YYYYMMDD_HH.pkl
# - Each file contains a dictionary with:
#     {
#         "date": datetime,
#         "fields": {key: np.ndarray}
#     }
# - Model predicts only a subset of fields
# - Time resolution:
#     - Input stepping: 12 hours
#     - Prediction lead time: 12 hours

input_path = Path("./era5/aifs_preprocessed")
output_path = Path("./HHO_AIFS_output")
output_path.mkdir(parents=True, exist_ok=True)

s3_ckpt = "s3://enw-04241552-kx1nks-shared/data/checkpoints/aifs_single_v0.2.1.ckpt"
local_ckpt = Path("./era5/aifs_single_v0.2.1.ckpt")
local_ckpt.parent.mkdir(parents=True, exist_ok=True)
if not local_ckpt.exists():
    fs = s3fs.S3FileSystem()
    fs.get(s3_ckpt, local_ckpt)
else:
    print("Checkpoint already exists:", local_ckpt)
#checkpoint = "./era5/aifs_single_v0.2.1.ckpt"

rollout_steps = 1

# define your date range here
start_date = datetime(2024, 5, 1, 12)
end_date   = datetime(2024, 5, 2, 0)

Checkpoint already exists: era5/aifs_single_v0.2.1.ckpt


In [5]:
import torch

torch.cuda.reset_peak_memory_stats()

runner = SimpleRunner(
    local_ckpt,
    device="cuda",
    precision="bfloat16",
)

print("After runner creation:")
print(torch.cuda.memory_allocated() / 1024**3, "GiB allocated")
print(torch.cuda.memory_reserved() / 1024**3, "GiB reserved")

After runner creation:
0.0 GiB allocated
0.0 GiB reserved


In [6]:
# -----------------------------
# MODEL
# -----------------------------

# Assumptions:
# - CUDA is available
# - Checkpoint is compatible with SimpleRunner
# - Inference is run in eval mode (no gradients)

print("Loading model...")

runner = SimpleRunner(local_ckpt, device="cuda", precision="bfloat16")
print(runner.precision)

Loading model...
bfloat16


## Main Processing Loop

This loop iterates over the specified date range and runs inference for each available input state.

Each iteration:
- Loads a state file
- Runs autoregressive rollouts
- Saves full trajectory output as a pickled file that will be regridded in postprocessing

In [7]:
current_date = start_date

while current_date <= end_date:

    timestamp = current_date.strftime("%Y%m%d_%H")
    file = input_path / f"{timestamp}.pkl"

    print("\n====================================")
    print("Running:", file)
    print("====================================")

    if not file.exists():
        print("Missing file, skipping:", file)
        current_date += timedelta(hours=6)
        continue

    with open(file, "rb") as f:
        current_state = pickle.load(f)

    torch.cuda.reset_peak_memory_stats()
    print("Before run:", torch.cuda.memory_allocated() / 1024**3)
    
    with torch.inference_mode():
        with torch.autocast("cuda", dtype=torch.bfloat16):
            gen = runner.run(input_state=current_state, lead_time=6)
    
            print("After gen created:", torch.cuda.memory_allocated() / 1024**3)
    
            state = next(gen)
    
            print("After first next:", torch.cuda.memory_allocated() / 1024**3)
            print("Peak:", torch.cuda.max_memory_allocated() / 1024**3)

    # trajectory = [current_state]

    # for step in range(rollout_steps):

    #     # -------------------------------------------------
    #     # IMPORTANT: do NOT use list(runner.run(...))[-1]
    #     # -------------------------------------------------
    #     with torch.inference_mode():
    #         with torch.autocast("cuda", dtype=torch.bfloat16):
    #             next_state = None
    
    #             for state in runner.run(
    #                 input_state=current_state,
    #                 lead_time=6,
    #             ):
    #                 if next_state is not None:
    #                     del next_state
    
    #                 next_state = state

    #     print(f"\n--- Step {step + 1} ---")
        # print_state(next_state)

        # new_fields = {}
        # predicted_keys = set(next_state["fields"].keys())

        # for key in current_state["fields"]:

        #     if key not in predicted_keys:
        #         new_fields[key] = current_state["fields"][key]
        #         continue

        #     prev = current_state["fields"][key]
        #     new = next_state["fields"][key]

        #     # Move torch tensors to CPU/numpy if needed
        #     if torch.is_tensor(prev):
        #         prev = prev.detach().cpu().numpy()

        #     if torch.is_tensor(new):
        #         new = new.detach().cpu().numpy()

        #     if new.ndim == 1:
        #         new = new[None, :]

        #     new_fields[key] = np.concatenate([prev[1:2], new], axis=0)

        # Explicitly remove old state references
        del current_state
        del next_state
        del state

        # current_state = {
        #     "date": trajectory[-1]["date"] + timedelta(hours=12),
        #     "fields": new_fields,
        # }

        # trajectory.append(current_state)

        torch.cuda.empty_cache()
        gc.collect()

    # print("\nSaving raw trajectory...")

    # out_file = output_path / f"HHO_aifs_rollout_{timestamp}.pkl"

    # with open(out_file, "wb") as f:
    #     pickle.dump(trajectory, f)

    # print("Saved:", out_file)

    # del trajectory
    # del current_state

    # torch.cuda.empty_cache()
    # gc.collect()

    current_date += timedelta(hours=12)


Running: era5/aifs_preprocessed/20240501_12.pkl
Before run: 0.0
After gen created: 0.0


OutOfMemoryError: CUDA out of memory. Tried to allocate 3.76 GiB. GPU 0 has a total capacity of 22.06 GiB of which 2.37 GiB is free. Including non-PyTorch memory, this process has 19.68 GiB memory in use. Of the allocated memory 19.31 GiB is allocated by PyTorch, and 73.61 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)